In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import re
from tqdm import tqdm
import time

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### Preprocessing and Tokenization

In [31]:
df = pd.read_csv("/content/drive/MyDrive/Deep Learning/Exp5/poems-100.csv")
df = df.dropna()

text = " <eol> ".join(df["text"].astype(str)).lower()
text = text.replace("’", "'")

text = re.sub(r"[^a-zA-Z'<>\s]", "", text)

words = text.split()

vocab = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

data = [word2idx[w] for w in words]
data_tensor = torch.tensor(data)

Vocabulary Size: 5479


### Numpy RNN Class

In [32]:
class NumpyRNN:
    def __init__(self, input_size, hidden_size, output_size, lr=0.001):
        self.hidden_size = hidden_size
        self.lr = lr

        self.U = np.random.randn(hidden_size, input_size) * np.sqrt(1 / input_size)
        self.W = np.random.randn(hidden_size, hidden_size) * np.sqrt(1 / hidden_size)
        self.V = np.random.randn(output_size, hidden_size) * np.sqrt(1 / hidden_size)

        self.b = np.zeros((hidden_size, 1))
        self.c = np.zeros((output_size, 1))

    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / np.sum(e_x)

    def forward(self, inputs):
        h = {}
        h[-1] = np.zeros((self.hidden_size, 1))
        y = {}
        p = {}

        for t in range(len(inputs)):
            x = inputs[t].reshape(-1, 1)
            h[t] = np.tanh(self.U @ x + self.W @ h[t-1] + self.b)
            y[t] = self.V @ h[t] + self.c
            p[t] = self.softmax(y[t])

        return h, p

    def backward(self, inputs, targets, h, p):
        dU = np.zeros_like(self.U)
        dW = np.zeros_like(self.W)
        dV = np.zeros_like(self.V)
        db = np.zeros_like(self.b)
        dc = np.zeros_like(self.c)

        dh_next = np.zeros((self.hidden_size, 1))
        loss = 0

        for t in reversed(range(len(inputs))):
            loss += -np.log(p[t][targets[t], 0] + 1e-9)

            dy = p[t].copy()
            dy[targets[t]] -= 1

            dV += dy @ h[t].T
            dc += dy

            dh = self.V.T @ dy + dh_next
            dh_raw = (1 - h[t] ** 2) * dh

            db += dh_raw
            dU += dh_raw @ inputs[t].reshape(1, -1)
            dW += dh_raw @ h[t-1].T

            dh_next = self.W.T @ dh_raw

        self.U -= self.lr * dU
        self.W -= self.lr * dW
        self.V -= self.lr * dV
        self.b -= self.lr * db
        self.c -= self.lr * dc

        return loss

### One-Hot Encoding on Numpy

In [33]:
def one_hot(index, vocab_size):
    vec = np.zeros((vocab_size,))
    vec[index] = 1
    return vec

sequence_length = 10
epochs = 10

np_onehot = NumpyRNN(vocab_size, 128, vocab_size)
np_onehot_loss = 0
np_onehot_time = 0
np_onehot_perp = 0
start = time.time()
for epoch in range(epochs):
    total_loss = 0
    iters = 0
    loop = tqdm(range(0, len(data)-sequence_length, sequence_length),
                desc=f"NumPy OneHot Epoch {epoch+1}")

    for i in loop:
        seq = data[i:i+sequence_length]
        targets = data[i+1:i+sequence_length+1]

        inputs = [one_hot(idx, vocab_size) for idx in seq]

        h, p = np_onehot.forward(inputs)
        loss = np_onehot.backward(inputs, targets, h, p)

        total_loss += loss
        iters += 1
        loop.set_postfix(loss=loss)

    avg_loss = total_loss/iters
    avg_perp = torch.exp(torch.tensor(avg_loss))
    np_onehot_loss += avg_loss
    print(f"Epoch {epoch+1} | " f"Avg Loss: {avg_loss:.4f} | " f"Perplexity: {avg_perp:.2f}")

np_onehot_time = time.time() - start
np_onehot_loss /= epochs
np_onehot_perp = torch.exp(torch.tensor(np_onehot_loss))

NumPy OneHot Epoch 1: 100%|██████████| 2765/2765 [04:31<00:00, 10.17it/s, loss=67.8]


Epoch 1 | Avg Loss: 69.6466 | Perplexity: 1766665812829903590944223002624.00


NumPy OneHot Epoch 2: 100%|██████████| 2765/2765 [04:25<00:00, 10.42it/s, loss=65.1]


Epoch 2 | Avg Loss: 64.4856 | Perplexity: 10133352479085306648937365504.00


NumPy OneHot Epoch 3: 100%|██████████| 2765/2765 [04:23<00:00, 10.48it/s, loss=64]


...

Epoch 8 | Avg Loss: 60.6450 | Perplexity: 217660874501173476390338560.00


NumPy OneHot Epoch 9: 100%|██████████| 2765/2765 [04:25<00:00, 10.43it/s, loss=61.4]


Epoch 9 | Avg Loss: 60.3491 | Perplexity: 161912256760796854202925056.00


NumPy OneHot Epoch 10: 100%|██████████| 2765/2765 [04:27<00:00, 10.35it/s, loss=61.1]

Epoch 10 | Avg Loss: 60.0774 | Perplexity: 123390679585304062099193856.00


### Embeddings on Numpy

In [34]:
embedding_dim = 100
embedding_matrix = np.random.randn(vocab_size, embedding_dim) * 0.01

np_embed = NumpyRNN(embedding_dim, 128, vocab_size)
np_emb_loss = 0
np_emb_time = 0
np_emb_perp = 0
start = time.time()
for epoch in range(epochs):
    total_loss = 0
    iters += 1
    loop = tqdm(range(0, len(data)-sequence_length, sequence_length),
                desc=f"NumPy Embed Epoch {epoch+1}")

    for i in loop:
        seq = data[i:i+sequence_length]
        targets = data[i+1:i+sequence_length+1]

        inputs = [embedding_matrix[idx] for idx in seq]

        h, p = np_embed.forward(inputs)
        loss = np_embed.backward(inputs, targets, h, p)

        total_loss += loss
        iters += 1
        loop.set_postfix(loss=loss)

    avg_loss = total_loss/iters
    avg_perp = torch.exp(torch.tensor(avg_loss))
    np_emb_loss += avg_loss
    print(f"Epoch {epoch+1} | " f"Avg Loss: {avg_loss:.4f} | " f"Perplexity: {avg_perp:.2f}")

np_emb_time = time.time() - start
np_emb_loss /= epochs
np_emb_perp = torch.exp(torch.tensor(np_emb_loss))

NumPy Embed Epoch 1: 100%|██████████| 2765/2765 [02:25<00:00, 19.07it/s, loss=68.1]


Epoch 1 | Avg Loss: 34.8926 | Perplexity: 1424444334316367.75


NumPy Embed Epoch 2: 100%|██████████| 2765/2765 [02:25<00:00, 19.02it/s, loss=65.6]


Epoch 2 | Avg Loss: 21.5609 | Perplexity: 2310948776.64


NumPy Embed Epoch 3: 100%|██████████| 2765/2765 [02:25<00:00, 19.04it/s, loss=64.1]


...

Epoch 8 | Avg Loss: 6.9184 | Perplexity: 1010.73


NumPy Embed Epoch 9: 100%|██████████| 2765/2765 [02:28<00:00, 18.66it/s, loss=61.3]


Epoch 9 | Avg Loss: 6.2162 | Perplexity: 500.82


NumPy Embed Epoch 10: 100%|██████████| 2765/2765 [02:33<00:00, 18.01it/s, loss=61.1]

Epoch 10 | Avg Loss: 5.6428 | Perplexity: 282.25


### Torch OneHot Model and Training

In [7]:
class OneHotRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size,  device=device)

In [15]:
def one_hot_torch(index):
    vec = torch.zeros(vocab_size)
    vec[index] = 1
    return vec

model_onehot = OneHotRNN(vocab_size, 128).to(device)
optimizer = optim.Adam(model_onehot.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

torch_onehot_loss = 0
torch_onehot_time = 0
torch_onehot_perp = 0
start = time.time()

for epoch in range(epochs):
    total_loss = 0
    iters = 0
    model_onehot.train()
    hidden = model_onehot.init_hidden(1)
    loop = tqdm(range(len(data)-sequence_length),
                desc=f"PyTorch OneHot {epoch+1}")

    for i in loop:
        seq = data[i:i+sequence_length]
        target = data[i+1:i+sequence_length+1]

        inputs = torch.stack([one_hot_torch(idx) for idx in seq])
        inputs = inputs.unsqueeze(0).to(device)
        target = torch.tensor(target).unsqueeze(0).to(device)

        # hidden = model_onehot.init_hidden(1)
        optimizer.zero_grad()

        output, hidden = model_onehot(inputs.float(), hidden)
        hidden = hidden.detach()

        loss = criterion(output.view(-1, vocab_size), target.view(-1))

        loss.backward()
        optimizer.step()

        total_loss +=loss.item()
        iters +=1

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss/iters
    avg_perp = torch.exp(torch.tensor(avg_loss))
    torch_onehot_loss += avg_loss
    print(f"Epoch {epoch+1} | " f"Avg Loss: {avg_loss:.4f} | " f"Perplexity: {avg_perp:.2f}")

torch_onehot_time = time.time() - start
torch_onehot_loss /= epochs
torch_onehot_perp = torch.exp(torch.tensor(torch_onehot_loss))
torch.save(model_onehot.state_dict(), "/content/drive/MyDrive/torch_onehot.pth")

PyTorch OneHot 1: 100%|██████████| 27649/27649 [01:54<00:00, 241.85it/s, loss=7.09]


Epoch 1 | Avg Loss: 6.4718 | Perplexity: 646.65


PyTorch OneHot 2: 100%|██████████| 27649/27649 [01:51<00:00, 248.23it/s, loss=7.45]


Epoch 2 | Avg Loss: 6.5849 | Perplexity: 724.05


PyTorch OneHot 3: 100%|██████████| 27649/27649 [01:50<00:00, 249.18it/s, loss=7.17]


...

Epoch 8 | Avg Loss: 6.2073 | Perplexity: 496.36


PyTorch OneHot 9: 100%|██████████| 27649/27649 [01:53<00:00, 243.17it/s, loss=6.86]


Epoch 9 | Avg Loss: 6.1419 | Perplexity: 464.94


PyTorch OneHot 10: 100%|██████████| 27649/27649 [01:51<00:00, 248.52it/s, loss=6.41]


Epoch 10 | Avg Loss: 6.0800 | Perplexity: 437.04


### Torch Embeddings Model and Training

In [16]:
class EmbeddingRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

In [17]:
model_embed = EmbeddingRNN(vocab_size, 100, 128).to(device)
optimizer = optim.Adam(model_embed.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

data_tensor = torch.tensor(data, device=device)

torch_embed_loss = 0
torch_embed_time = 0
torch_embed_perp = 0
start = time.time()

for epoch in range(epochs):
    total_loss = 0
    iters = 0
    model_embed.train()
    hidden = model_embed.init_hidden(1)
    loop = tqdm(range(len(data_tensor)-sequence_length),
                desc=f"PyTorch Embed {epoch+1}")

    for i in loop:
        seq = data_tensor[i:i+sequence_length].unsqueeze(0).to(device)
        target = data_tensor[i+1:i+sequence_length+1].unsqueeze(0).to(device)

        # hidden = model_embed.init_hidden(1)
        optimizer.zero_grad()

        output, hidden = model_embed(seq, hidden)
        hidden = hidden.detach()
        loss = criterion(output.view(-1, vocab_size), target.view(-1))

        loss.backward()
        optimizer.step()

        total_loss +=loss.item()
        iters +=1

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss/iters
    avg_perp = torch.exp(torch.tensor(avg_loss))
    torch_embed_loss += avg_loss
    print(f"Epoch {epoch+1} | " f"Avg Loss: {avg_loss:.4f} | " f"Perplexity: {avg_perp:.2f}")

torch_embed_loss /= epochs
torch_embed_perp = torch.exp(torch.tensor(torch_embed_loss))
torch_embed_time = time.time() - start
torch.save(model_embed.state_dict(), "/content/drive/MyDrive/torch_embed.pth")

PyTorch Embed 1: 100%|██████████| 27649/27649 [01:43<00:00, 267.25it/s, loss=7.17]


Epoch 1 | Avg Loss: 6.2042 | Perplexity: 494.83


PyTorch Embed 2: 100%|██████████| 27649/27649 [01:40<00:00, 274.00it/s, loss=6.79]


Epoch 2 | Avg Loss: 6.4125 | Perplexity: 609.39


PyTorch Embed 3: 100%|██████████| 27649/27649 [01:40<00:00, 273.78it/s, loss=6.51]


...

Epoch 8 | Avg Loss: 5.8538 | Perplexity: 348.55


PyTorch Embed 9: 100%|██████████| 27649/27649 [01:58<00:00, 232.77it/s, loss=6.61]


Epoch 9 | Avg Loss: 5.7788 | Perplexity: 323.39


PyTorch Embed 10: 100%|██████████| 27649/27649 [01:50<00:00, 251.05it/s, loss=7.18]


Epoch 10 | Avg Loss: 5.7121 | Perplexity: 302.50


In [50]:
def generate_text_numpy(rnn, start_word, length=20, embedding_matrix=None):
    current_idx = word2idx[start_word]
    generated = [start_word]
    h_prev = np.zeros((rnn.hidden_size, 1))

    for _ in range(length):
        if embedding_matrix is not None:
            x = embedding_matrix[current_idx].reshape(-1,1)
        else:
            x = one_hot(current_idx, vocab_size).reshape(-1,1)

        h = np.tanh(rnn.U @ x + rnn.W @ h_prev + rnn.b)
        y = rnn.V @ h + rnn.c
        probs = np.exp(y) / np.sum(np.exp(y))

        current_idx = np.random.choice(range(vocab_size), p=probs.ravel())
        generated.append(idx2word[current_idx])

        h_prev = h

    return " ".join(generated)


In [51]:
def generate_text_pytorch(model, start_word, length=20):
    model.eval()
    batch_size = 1
    hidden = model.init_hidden(batch_size)

    use_embedding = hasattr(model, "embedding")

    if use_embedding:
        current_idx = torch.tensor([[word2idx[start_word]]]).to(device)
    else:
        current_idx = torch.zeros(1, 1, len(word2idx)).to(device)
        current_idx[0, 0, word2idx[start_word]] = 1

    generated = [start_word]

    for _ in range(length):
        with torch.no_grad():
            output, hidden = model(current_idx, hidden)
            probs = torch.softmax(output[0, -1], dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            generated.append(idx2word[next_idx])

            if use_embedding:
                current_idx = torch.tensor([[next_idx]]).to(device)
            else:
                current_idx = torch.zeros(1, 1, len(word2idx)).to(device)
                current_idx[0, 0, next_idx] = 1

    return " ".join(generated)

### Text Generation using each Model

In [46]:
print("\nNumPy RNN One-Hot Sample:")
print(generate_text_numpy(np_onehot, start_word="love", length=20))


NumPy RNN One-Hot Sample:
love good <eol> comes can could and those suckled smoke his all waists a to <eol> the me mockers the am


In [47]:
print("\nNumPy RNN Embedding Sample:")
print(generate_text_numpy(np_embed, start_word="love", length=20, embedding_matrix=embedding_matrix))


NumPy RNN Embedding Sample:
love making <eol> countenance sweetheart like i hands funeral all even the <eol> knows over clear words me in <eol> light


In [48]:
print("\nPyTorch RNN One-Hot Sample:")
print(generate_text_pytorch(model_onehot, start_word="love"))


PyTorch RNN One-Hot Sample:
love like a creation while <eol> by one who goes to over <eol> this let it first eyes our with <eol>


In [49]:
print("\nPyTorch RNN Embedding Sample:")
print(generate_text_pytorch(model_embed, start_word="love"))


PyTorch RNN Embedding Sample:
love that when they have cause thy he who i was three country give what if a single faces of <eol>


### Final Comparison

In [43]:
comparison = {
    'Model': ['Numpy_Onehot', 'Numpy_Embed', 'Torch_Onehot', 'Torch_Embed'],
    'Loss': [np_onehot_loss, np_emb_loss, torch_onehot_loss, torch_embed_loss],
    'Time': [np_onehot_time, np_emb_time, torch_onehot_time, torch_embed_time],
    'Perplexity': [np_onehot_perp.item(), np_emb_perp.item(), torch_onehot_perp.item(), torch_embed_perp.item()]
}
table = pd.DataFrame(comparison)
table

,Model,Loss,Time,Perplexity
0,Numpy_Onehot,62.476335,2662.019616,1.358713e+27
1,Numpy_Embed,13.091201,1469.983271,4.846590e+05
2,Torch_Onehot,6.363018,1115.762370,5.799941e+02
3,Torch_Embed,6.071285,1068.717192,4.332372e+02
